# Additional sources in Jupyter Notebook
### Created by `Siddhant Mahajan`

This notebook contains data, code and literature from various sources including and extending beyond:
 - Google (And AI assisted searches)
 - Bing
 - Copilot Searches
 - Generative AI tool outputs
 - Research papers that came as results when I searched for various parameters
 - PPTs uploaded to sharing websites publically

### Generating VLE values 

Ethanol, water Antoine Values: https://myengineeringtools.com/Data_Diagrams/Antoine_Law_Coefficients.html

NRTL interaction parameter values: https://onlinelibrary.wiley.com/doi/pdf/10.1002/9781118477304.app2?msockid=00a9a1ee16cf6f3e25d2b7b717296eff


In [98]:
import numpy as np
import pandas as pd

In [ ]:
def antoine_equation_for_vapor_pressure(A, B, C, T):
    # returns vapor pressure in mmHG calculatded based of antoine values and Temp in deg C
    return 10**(A - B / (T + C))

def nrtl_gamma(x1, x2, tau12, tau21, alpha):
    # gamma values calculation for NRTL model
    G12 = np.exp(-alpha * tau12)
    G21 = np.exp(-alpha * tau21)
    gamma1 = np.exp(x2**2 * (tau21 * G21) / (x1 + x2 * G21)**2)
    gamma2 = np.exp(x1**2 * (tau12 * G12) / (x2 + x1 * G12)**2)
    
    return gamma1, gamma2

def nrtl_tau(a, b, T):
    return a + b / T

def vle_water_ethanol(T_C, x1):
    T = T_C + 273.15
    
    # NRTL interaction parameters
    alpha = 0.3
    a12, a21 = -0.8009, 3.4578
    b12, b21 = 246.2, -586.1
    
    # antoine constants ethanol
    A_ethanol, B_ethanol, C_ethanol = 8.2133, 1652.05, 231.48
    # antoine constants water
    A_water, B_water, C_water = 8.07131, 1730.63, 233.426
    
    x2 = 1 - x1
    tau12 = nrtl_tau(a12, b12, T)
    tau21 = nrtl_tau(a21, b21, T)
    gamma1, gamma2 = nrtl_gamma(x1, x2, tau12, tau21, alpha)
    P1sat = antoine_equation_for_vapor_pressure(A_ethanol, B_ethanol, C_ethanol, T_C)
    P2sat = antoine_equation_for_vapor_pressure(A_water, B_water, C_water, T_C)
    P = x1 * gamma1 * P1sat + x2 * gamma2 * P2sat
    y1 = x1 * gamma1 * P1sat / P
    y2 = x2 * gamma2 * P2sat / P
    return P, T, x1, x2, y1, y2

# Example calculation at azeotropic point

T_azeo = 78.2  # Celsius
x1_azeo = 0.89 # Ethanol
# x1_azeo = 0.25
P, T, x1, x2, y1, y2 = vle_water_ethanol(T_azeo, x1_azeo)
print(f"P = {P:.2f} mmHg, T = {T:.2f} K, x1 = {x1:.3f}, x2 = {x2:.3f}, y1 = {y1:.3f}, y2 = {y2:.3f}")


P = 715.91 mmHg, T = 351.35 K, x1 = 0.890, x2 = 0.110, y1 = 0.953, y2 = 0.047


In [99]:
df = pd.DataFrame(columns=['Pressure', 'Temperature', 'x1', 'y1'])

In [100]:
df.shape

(0, 4)

In [103]:
for k in range(60,100):
    for i in range(0,100):
        j = i/100
        P, T, x1, x2, y1, y2 = vle_water_ethanol(k, j)
        if round(x1-y1, 2) == 0 and j!=0:
            # print(f"P = {P:.2f} mmHg, T = {T:.2f} K, x1 = {x1:.3f}, x2 = {x2:.3f}, y1 = {y1:.3f}, y2 = {y2:.3f}")
            df.loc[df.shape[0]] = [P,T,x1,y1]

In [104]:
df

,Pressure,Temperature,x1,y1
0,336.647251,333.15,0.92,0.966363
1,338.205898,333.15,0.93,0.970736
2,339.836648,333.15,0.94,0.975065
3,341.539863,333.15,0.95,0.979348
4,343.315926,333.15,0.96,0.983582
...,...,...,...,...
315,1593.098559,372.15,0.95,0.979612
316,1601.355313,372.15,0.96,0.983813
317,1609.989416,372.15,0.97,0.987955
318,1619.002272,372.15,0.98,0.992035


In [106]:
df.to_csv('Data_csv.csv', sep=",", index=False)